# 02 · Spark ML at Scale with MLflow

**Goal:** train a model with Spark ML on the full dataset, choose its settings with cross-validation, track every run
in MLflow, and register the best model in Unity Catalog.

**Building blocks used here**

| Block | Role |
|---|---|
| `VectorAssembler` | Puts all feature columns into one `features` vector (Spark ML models need this) |
| `LogisticRegression`, `RandomForestClassifier` | The models (Estimators: `.fit()` learns, returns a model) |
| `Pipeline` | Chains the steps so the same preparation runs at training and prediction time |
| `CrossValidator` + `ParamGridBuilder` | Tries several settings, each scored on several data folds |
| `BinaryClassificationEvaluator` | Scores predictions with AUC (1.0 = perfect, 0.5 = guessing) |

In [ ]:
%run ./Includes/_config

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from mlflow.models.signature import infer_signature

df = spark.table(TABLE)
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,} rows   Test: {test_df.count():,} rows")

assembler = VectorAssembler(inputCols=FEATURES, outputCol="features")
evaluator = BinaryClassificationEvaluator(labelCol=LABEL, metricName="areaUnderROC")

## 1. Cross-validated logistic regression, tracked in MLflow
The grid tries two regularisation strengths. With 3 folds that is 6 training runs, which Spark spreads over the cluster.

In [ ]:
lr = LogisticRegression(labelCol=LABEL, featuresCol="features", maxIter=50)
grid = ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.05]).build()
cv = CrossValidator(estimator=Pipeline(stages=[assembler, lr]), estimatorParamMaps=grid,
                    evaluator=evaluator, numFolds=3, seed=42)

with mlflow.start_run(run_name="spark_logistic_regression_cv") as lr_run:
    cv_model = cv.fit(train_df)
    lr_auc = evaluator.evaluate(cv_model.transform(test_df))
    best_reg = grid[int(max(range(len(grid)), key=lambda i: cv_model.avgMetrics[i]))][lr.regParam]
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("best_regParam", best_reg)
    for params, score in zip(grid, cv_model.avgMetrics):
        mlflow.log_metric(f"cv_auc_regParam_{params[lr.regParam]}", score)
    mlflow.log_metric("test_auc", lr_auc)
print(f"Logistic regression  test AUC = {lr_auc:.4f}  (best regParam = {best_reg})")

## 2. Random forest for comparison

In [ ]:
rf = RandomForestClassifier(labelCol=LABEL, featuresCol="features", numTrees=30, maxDepth=6, seed=42)
with mlflow.start_run(run_name="spark_random_forest") as rf_run:
    rf_model = Pipeline(stages=[assembler, rf]).fit(train_df)
    rf_auc = evaluator.evaluate(rf_model.transform(test_df))
    mlflow.log_params({"model": "RandomForest", "numTrees": 30, "maxDepth": 6})
    mlflow.log_metric("test_auc", rf_auc)
print(f"Random forest        test AUC = {rf_auc:.4f}")

## 3. Register the better model in Unity Catalog
Unity Catalog models need a **signature** (input and output columns), inferred here from a few example rows.

In [ ]:
best_model, best_name, best_auc = (cv_model.bestModel, "logistic_regression", lr_auc) if lr_auc >= rf_auc else (rf_model, "random_forest", rf_auc)
model_name = f"{CATALOG}.{SCHEMA}.diabetes_spark_model"

example = train_df.select(FEATURES).limit(5)
signature = infer_signature(example.toPandas(), best_model.transform(train_df.limit(5)).select("prediction").toPandas())

with mlflow.start_run(run_name=f"register_{best_name}"):
    mlflow.log_metric("test_auc", best_auc)
    info = mlflow.spark.log_model(best_model, "model", signature=signature,
                                  input_example=example.toPandas(), registered_model_name=model_name)
print(f"Registered {best_name} (AUC {best_auc:.4f}) as {model_name}")

## 4. Score the whole test set with the registered model
A Spark ML model is loaded back as a Spark pipeline with `mlflow.spark.load_model`. Its `.transform()` scores every row
as a normal distributed Spark job. (For scikit-learn and other Python models, `mlflow.pyfunc.spark_udf` does the same job.)

In [ ]:
client = mlflow.MlflowClient()
latest = max(int(v.version) for v in client.search_model_versions(f"name = '{model_name}'"))
loaded = mlflow.spark.load_model(f"models:/{model_name}/{latest}")
scored = loaded.transform(test_df)
display(scored.groupBy(LABEL, "prediction").count().orderBy(LABEL, "prediction"))

## Key points
- Spark ML trains on the **whole** table, spread across the cluster.
- `CrossValidator` picks settings fairly by scoring each one on several folds.
- Every run is in the **SWSAcademy_Advanced_ML_Concepts** experiment; open it from the Experiments page to compare.
- The best model is a versioned asset in Unity Catalog, ready for batch scoring or serving.